# Honeypot log analysis

Rebuilds the numbers and charts in [README_REPORT.md](../README_REPORT.md) from the visit logs
that `deploy/daily-logs.sh` copies into `~/honeypot-logs`. Run the cells from the top. The last
section checks every number against the report.

**Handling the logs safely**

- **IPs:** the loader turns each address into a masked label (first two numbers) and an ID made
  with a random key that only exists while the kernel runs. IPs can still be counted and matched
  across periods, but no full address is ever kept or shown.
- **Passwords:** read once, only to find my own test visits (password `not-on-the-list`), then
  dropped.
- **Commands** appear only in the playbook table: defanged (`hxxp`, `[.]`), shortened, and never
  as links. **Usernames** appear only when several IPs tried them, as in the daily summary,
  because a one-off could be a password typed into the wrong box.
- **Nothing from the logs is ever fetched, opened or run.**
- **Git:** outputs hold log data. `.gitattributes` strips them from this notebook on every commit
  and `analysis/figures/` is ignored, but check `git diff --cached` before committing anyway. On a
  fresh clone, set the filter up first:
  `git config filter.strip-notebook-output.clean "jq --indent 1 -f analysis/strip-outputs.jq"`
  and `git config filter.strip-notebook-output.required true`.
- Never display `visits_all`, `visits`, `after`, `sessions` or `cmds` directly: they still hold
  commands and usernames.

In [ ]:
import hashlib
import hmac
import html
import json
import re
import secrets
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_colwidth", 100)

# The repository root, found from wherever the notebook runs.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Cargo.toml").exists())
LOG_DIR = Path.home() / "honeypot-logs"


def utc(text):
    return datetime.fromisoformat(text).replace(tzinfo=timezone.utc)


# From the deployment record. All times are UTC.
STAGE6_OPEN = utc("2026-11-02 00:17:40")  # fake login only, on the old server and IP
PORT_CLOSED = utc("2026-11-03 01:00:16")  # closed for the reinstall
CHANGE = utc("2026-11-03 01:29:49")  # reopened with the telnet opening and shell, on a new IP
REPORT_END = utc("2026-11-06 15:34:11")  # the last visit counted in README_REPORT.md

# The report's "before" column is the daily summary run at 00:59 on Nov 3: the 24 hours before it.
BASELINE_END = utc("2026-11-03 00:59:00")
BASELINE_START = BASELINE_END - timedelta(hours=24)

# Set END = None to include logs copied since the report. The checks at the bottom then won't match.
END = REPORT_END

# "light" for the report and site images, "dark" to match a dark editor.
THEME = "light"

## Load the logs

Full IPs and passwords only exist inside `load_visits`. What comes out has a masked label and an
opaque ID for each IP, and no passwords.

In [ ]:
# A new random key each time the kernel starts. Without it, an ID can't be turned back into an
# address, even by trying every IPv4 address.
_KEY = secrets.token_bytes(32)


def opaque_id(text):
    return hmac.new(_KEY, text.encode(), hashlib.sha256).hexdigest()[:16]


def mask_ip(ip):
    parts = ip.split(".")
    return f"{parts[0]}.{parts[1]}.x.x" if len(parts) == 4 else "(not IPv4)"


def url_port(url):
    """The port a download address names, read from its text. Nothing is contacted."""
    if "://" not in url:
        return None  # a bare host, as tftp takes it
    try:
        parts = urlsplit(url)
        return parts.port or {"http": 80, "https": 443}.get(parts.scheme.lower())
    except ValueError:
        return None


def load_visits(log_dir):
    files = sorted(log_dir.glob("visits-*.jsonl"))
    raw, skipped = [], 0
    for path in files:
        with path.open(encoding="utf-8", errors="replace") as f:
            for line in f:
                try:
                    raw.append(json.loads(line))
                except json.JSONDecodeError:
                    skipped += 1  # a line cut off while the log was being copied
    # The one use of passwords: my own test visits used this one, so their IPs are left out.
    own = {v["source_ip"] for v in raw if v.get("password") == "not-on-the-list"}
    rows = []
    for v in raw:
        ip = v["source_ip"]
        shell = v.get("shell") or {}
        urls = shell.get("urls") or []
        rows.append(
            {
                "t": v["time"],
                "time": pd.Timestamp(v["time"], unit="s", tz="UTC"),
                "ip_id": opaque_id(ip),
                "ip_masked": mask_ip(ip),
                "excluded": ip.startswith("127.") or ip in own,
                "outcome": v["outcome"],
                "username": v.get("username"),
                "telnet_answered": bool(v.get("telnet")),
                "ended": shell.get("ended"),
                "commands": shell.get("commands") or [],
                "url_ids": [opaque_id(u) for u in urls],
                "url_ports": [url_port(u) for u in urls],
            }
        )
    print(
        f"Read {len(raw):,} lines from {len(files)} files, skipped {skipped} unreadable ones, "
        f"and found {len(own)} of my own test IPs to leave out."
    )
    return pd.DataFrame(rows)


visits_all = load_visits(LOG_DIR)
assert not {"source_ip", "password", "telnet"} & set(visits_all.columns)
visits = visits_all[~visits_all["excluded"]]

## Periods and shared rules

The same rules as the daily summary and the report: a visit is silent when it sent no username
line, and a retry-pattern IP has 5 or more silent visits about 13 seconds apart.

In [ ]:
in_range = visits["time"] <= END if END else pd.Series(True, index=visits.index)
after = visits[(visits["time"] >= CHANGE) & in_range]
before = visits[(visits["time"] >= BASELINE_START) & (visits["time"] < BASELINE_END)]
first_day = after[after["time"] < CHANGE + timedelta(hours=24)]
sessions = after[after["outcome"] == "logged_in"].copy()


def silent(df):
    """Visits that sent nothing at all."""
    return df[df["username"].isna()]


def sent_login(df):
    return df[df["outcome"].isin(["attempted", "logged_in"])]


def pct(part, whole, digits=0):
    value = round(100 * part / whole, digits)
    return int(value) if digits == 0 else value


def retry_ips(df):
    """IPs with 5+ silent visits whose median gap is 12 to 15 seconds."""
    found = set()
    for ip, times in silent(df).groupby("ip_id")["t"]:
        t = sorted(times)
        if len(t) < 5:
            continue
        gaps = sorted(b - a for a, b in zip(t, t[1:]))
        if 12 <= gaps[len(gaps) // 2] <= 15:
            found.add(ip)
    return found


def printable(text):
    return "".join(ch for ch in text if " " <= ch <= "~")


def safe_usernames(df, min_ips=2):
    """Usernames tried by at least min_ips IPs, with control characters removed."""
    named = df[df["username"].fillna("") != ""]
    table = named.groupby("username").agg(attempts=("ip_id", "size"), ips=("ip_id", "nunique"))
    table = table[table["ips"] >= min_ips].sort_values(["ips", "attempts"], ascending=False)
    table.index = [printable(u)[:32] for u in table.index]
    table.index.name = "username"
    return table


hours_live = (after["time"].max() - CHANGE).total_seconds() / 3600
print(
    f"Before: {len(before):,} visits in the 24 hours to {BASELINE_END:%b %d %H:%M} UTC. "
    f"After: {len(after):,} visits over {hours_live:.0f} hours, {len(sessions)} of them shell sessions."
)

## Sort commands into playbook steps

Each command line gets one family, checked in a fixed order, which is how the report counted
them. Removing rivals and cleaning up are found inside download lines, so they're extra flags.

In [ ]:
MENU_ESCAPE = {"enable", "system", "shell", "sh", "linuxshell", "enablelinuxshell"}
# Some bots end these words with a NUL byte, as in Mirai's string tables. The log stores it as \0.
MENU_ESCAPE |= {w + "\\0" for w in ["enable", "system", "shell", "sh", "linuxshell"]}
ROUTER_CLI = {"config terminal", "start", "su"}
LOOK_AROUND = {"cat /proc", "uname -m", "cat /etc/passwd", "id", "busybox", "/bin/busybox"}
CHECKS = {"writable_probe", "busybox_tag", "echo_marker", "shell_test"}


def family(cmd):
    if cmd == "echo SHELL_TEST":
        return "shell_test"
    if cmd.startswith(">/var/run/.x&&"):
        return "writable_probe"
    if re.search(r"wget|curl|tftp", cmd):
        return "download"
    if re.match(r"/bin/busybox [A-Za-z0-9]", cmd):
        return "busybox_tag"
    if cmd.startswith("echo -e") or cmd == "echo PAYLOAD_WORKING":
        return "echo_marker"
    if cmd in MENU_ESCAPE:
        return "menu_escape"
    if cmd in ROUTER_CLI:
        return "router_cli"
    if re.fullmatch(r"ping ?; ?sh", cmd):
        return "ping_injection"
    if cmd in LOOK_AROUND or re.match(r"cat /proc/mounts|ls /home", cmd):
        return "look_around"
    return "other"


RIVALS = re.compile(r"for p in ([^;]*);\s*do\s+killall -9 \$p")
CLEAN_UP = re.compile(r"\bsh (\S+);\s*rm -f \1")

cmds = sessions[["ip_id", "commands"]].explode("commands").dropna(subset=["commands"])
cmds = cmds.rename(columns={"commands": "cmd"})
cmds["session"] = cmds.index
cmds["family"] = cmds["cmd"].map(family)
cmds["remove_rivals"] = cmds["cmd"].map(lambda c: RIVALS.search(c) is not None)
cmds["clean_up"] = cmds["cmd"].map(lambda c: CLEAN_UP.search(c) is not None)

families = cmds.groupby("session")["family"].agg(set).reindex(sessions.index)
sessions["families"] = families.map(lambda f: f if isinstance(f, set) else set())
sessions["ran_check"] = sessions["families"].map(lambda f: bool(f & CHECKS))
sessions["downloaded"] = sessions["url_ids"].map(len) > 0
print(f"{len(cmds):,} commands, {cmds['cmd'].nunique()} different, in {cmds['session'].nunique()} sessions.")

## Defanging, for the one table that shows commands

In [ ]:
IPV4 = re.compile(r"\b(\d{1,3})\.(\d{1,3})\.\d{1,3}\.\d{1,3}\b")
# Long letter-and-digit path parts look like campaign IDs, which could identify this honeypot.
CAMPAIGN_ID = re.compile(r"(?<=/)(?=[A-Za-z0-9]*\d)(?=[A-Za-z0-9]*[A-Za-z])[A-Za-z0-9]{8,}(?=/)")


def defang(text, limit=90):
    r"""Make a logged command safe to show: no full IPs, no working links, no control characters.

    The log stores commands escaped (Rust's escape_debug): backslashes doubled, quotes escaped,
    and control characters written as \0, \t or \u{...}. Undoing the first two shows the line as
    the bot sent it, while control characters stay visible as text.
    """
    text = re.sub(r"\\([\\'\"])", r"\1", text)
    text = printable(text)
    text = IPV4.sub(lambda m: f"{m[1]}[.]{m[2]}[.]x[.]x", text)
    text = re.sub(r"(?i)\bhttp", "hxxp", text)
    text = re.sub(r"(hxxps?://)([A-Za-z0-9.-]+)", lambda m: m[1] + m[2].replace(".", "[.]"), text)
    text = CAMPAIGN_ID.sub("<id>", text)
    return text if len(text) <= limit else text[: limit - 1] + "…"


def example_for(key, lines):
    common = lines.value_counts().index[0]
    if key == "remove_rivals":
        return defang(RIVALS.search(common)[0])
    if key == "clean_up":
        return defang(re.sub(r"\S+\.sh", "<name>.sh", CLEAN_UP.search(common)[0]))
    return defang(common)

## Chart style

Thin marks, one hue for a single series, a validated blue and orange pair when two series are
compared, and hairline grids. Each chart is followed by its numbers as a table.

In [ ]:
TOKENS = {
    "light": {"surface": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781",
              "grid": "#e1e0d9", "axis": "#c3c2b7", "s1": "#2a78d6", "s2": "#eb6834",
              "wash": "rgba(11,11,11,0.06)"},
    "dark": {"surface": "#1a1a19", "ink": "#f0efec", "ink2": "#c3c2b7", "muted": "#898781",
             "grid": "#2c2c2a", "axis": "#383835", "s1": "#3987e5", "s2": "#d95926",
             "wash": "rgba(255,255,255,0.08)"},
}
T = TOKENS[THEME]
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
FIGURES = {}


def style(fig, title, subtitle=None, height=420):
    title_spec = {"text": title, "x": 0.01, "xanchor": "left", "font": {"size": 18, "color": T["ink"]}}
    if subtitle:
        title_spec["subtitle"] = {"text": subtitle, "font": {"size": 13, "color": T["ink2"]}}
    axis = {"gridcolor": T["grid"], "gridwidth": 1, "linecolor": T["axis"], "zeroline": False,
            "tickfont": {"color": T["muted"]}, "title": {"font": {"color": T["ink2"]}}}
    fig.update_layout(
        title=title_spec,
        height=height,
        paper_bgcolor=T["surface"],
        plot_bgcolor=T["surface"],
        font={"family": FONT, "color": T["ink"], "size": 13},
        margin={"l": 24, "r": 48, "t": 110 if subtitle else 70, "b": 48},
        hoverlabel={"bgcolor": T["surface"], "bordercolor": T["axis"],
                    "font": {"color": T["ink"], "family": FONT}},
        legend={"orientation": "h", "x": 0, "y": 1.0, "yanchor": "bottom", "font": {"color": T["ink2"]}},
        barcornerradius=4,
    )
    fig.update_xaxes(**axis)
    fig.update_yaxes(**axis)
    return fig


def show(fig, name):
    FIGURES[name] = fig
    fig.show(config={"displaylogo": False,
                     "toImageButtonOptions": {"format": "png", "scale": 2, "filename": name}})


def naive(moment):
    """Plain UTC time for a chart axis."""
    return pd.Timestamp(moment).tz_convert(None)

## 1. From connection to download

In [ ]:
with_command = sessions["commands"].map(len) > 0
funnel = pd.DataFrame({
    "stage": ["Connections", "Sent a login", "Got into the shell", "Sent a command",
              "Ran a real-shell check", "Reached a download"],
    "count": [len(after), len(sent_login(after)), len(sessions), int(with_command.sum()),
              int(sessions["ran_check"].sum()), int(sessions["downloaded"].sum())],
})
funnel["share of connections"] = (100 * funnel["count"] / len(after)).round(1).astype(str) + "%"
checked_then_downloaded = int((sessions["ran_check"] & sessions["downloaded"]).sum())

fig = go.Figure(go.Funnel(
    y=funnel["stage"], x=funnel["count"], width=0.55, cliponaxis=False,
    texttemplate="%{value:,}  ·  %{percentInitial:.1%}", textposition="outside", textfont={"color": T["ink"]},
    marker={"color": T["s1"]},
    connector={"fillcolor": T["wash"], "line": {"width": 0}},
    hovertemplate="%{y}: %{x:,}, %{percentInitial:.1%} of connections<extra></extra>",
))
style(fig, "Most bots that got in left before downloading anything",
      f"Nov 3–6. The last two steps overlap rather than nest: {checked_then_downloaded} of the "
      f"{funnel['count'].iloc[-1]} sessions that reached a download ran a check first.")
fig.update_yaxes(showgrid=False, showline=False, ticks="")
fig.update_xaxes(visible=False)
fig.update_layout(margin={"r": 150})  # room for the labels beside the widest bars
show(fig, "1-funnel")
display(funnel)

## 2. Visits per hour, before and after the change

In [ ]:
span = visits[(visits["time"] >= STAGE6_OPEN) & in_range]
hourly = span.set_index("time").resample("1h").size().rename("visits")
hourly.index = hourly.index.tz_convert(None)

fig = go.Figure(go.Scatter(
    x=hourly.index, y=hourly.values, mode="lines", line={"color": T["s1"], "width": 2},
    hovertemplate="%{x|%b %d, %H:00} UTC: %{y} visits<extra></extra>",
))
fig.add_vrect(x0=naive(PORT_CLOSED), x1=naive(CHANGE), fillcolor=T["wash"], line_width=0, layer="below")
fig.add_shape(type="line", x0=naive(CHANGE), x1=naive(CHANGE), y0=0, y1=1, yref="paper",
              line={"color": T["ink2"], "width": 1})
for text, anchor in [("← Fake login only, old IP  ", "right"), ("  Telnet opening and fake shell, new IP →", "left")]:
    fig.add_annotation(x=naive(CHANGE), y=1, yref="paper", yanchor="bottom", xanchor=anchor,
                       text=text, showarrow=False, font={"color": T["ink2"], "size": 12})
style(fig, "Visits per hour, before and after the change",
      "UTC, from the fake login going live on Nov 2. The thin shaded strip is the 30 minutes the "
      "port was closed for the reinstall.")
fig.update_xaxes(showgrid=False, showline=True)
fig.update_yaxes(title_text="Visits per hour", rangemode="tozero")
show(fig, "2-visits-per-hour")

per_day = after.groupby(after["time"].dt.date).apply(
    lambda g: pd.Series({"visits": len(g), "silent": len(silent(g)),
                         "silent %": pct(len(silent(g)), len(g), 1), "shell sessions": int((g["outcome"] == "logged_in").sum())}),
    include_groups=False,
)
per_day.index.name = "UTC day, after the change"
display(per_day)

## 3. Silent visits before and after the change

The IP address changed at the same moment as the telnet opening, so this can't separate the two.

In [ ]:
retry_before = retry_ips(before)
seen_both = set(before["ip_id"]) & set(after["ip_id"])
before_no_retry = before[~before["ip_id"].isin(retry_before)]
retry_after = after[after["ip_id"].isin(retry_before)]
b_both, a_both = before[before["ip_id"].isin(seen_both)], after[after["ip_id"].isin(seen_both)]

silent_table = pd.DataFrame({
    "group": ["All visits", f"Without the {len(retry_before)} retry IPs", f"The {len(seen_both)} IPs seen in both"],
    "before silent": [len(silent(before)), len(silent(before_no_retry)), len(silent(b_both))],
    "before visits": [len(before), len(before_no_retry), len(b_both)],
    "after silent": [len(silent(after)), len(silent(retry_after)) if len(retry_after) else None, len(silent(a_both))],
    "after visits": [len(after), len(retry_after) if len(retry_after) else None, len(a_both)],
})
silent_table["before %"] = [pct(s, n) for s, n in zip(silent_table["before silent"], silent_table["before visits"])]
silent_table["after %"] = pd.array(
    [pct(s, n) if pd.notna(n) and n else None for s, n in zip(silent_table["after silent"], silent_table["after visits"])],
    dtype="Int64",
)

fig = go.Figure()
# Narrow bars side by side, 0.08 of each category wide with a small gap between them.
for period, color, label, offset in [("before", T["s1"], "Before: 24 hours, fake login only", -0.09),
                                     ("after", T["s2"], "After: Nov 3–6, telnet opening and shell", 0.01)]:
    values = [int(p) if pd.notna(p) else None for p in silent_table[f"{period} %"]]
    fig.add_bar(
        x=silent_table["group"], y=values, name=label, marker={"color": color}, width=0.08, offset=offset,
        text=[f"{p}%" if p is not None else "" for p in values],
        textposition="outside", textfont={"color": T["ink"]}, cliponaxis=False,
        customdata=silent_table[[f"{period} silent", f"{period} visits"]],
        hovertemplate=label + "<br>%{x}: %{y}% silent (%{customdata[0]} of %{customdata[1]} visits)<extra></extra>",
    )
if not len(retry_after):
    fig.add_annotation(x=silent_table["group"][1], y=1, xshift=3, text="never came back", showarrow=False,
                       xanchor="left", yanchor="bottom", font={"color": T["muted"], "size": 12})
b, a = silent_table["before %"][0], silent_table["after %"][0]
style(fig, f"Silent visits fell from {b}% to {a}% after the change",
      "But the IP address changed at the same moment, so the telnet opening isn't proven to be the cause.",
      height=440)
fig.update_yaxes(range=[0, 100], ticksuffix="%", showline=False)
fig.update_xaxes(showgrid=False, showline=True)
show(fig, "3-silent-before-after")
display(silent_table)

## 4. The post-login playbook, in attack order

The chart counts sessions. The table also gives command lines and IPs, which are the numbers in
the report's playbook table. Commands in it are defanged examples, the most common line in each
step.

In [ ]:
STEPS = [
    ("login", "Log in with factory defaults", "T1110.001, T1078.001"),
    ("menu_escape", "Break out of a router's limited menu", "T1059.008, T1059.004"),
    ("router_cli", "Try a router CLI", "T1059.008"),
    ("ping_injection", "Inject a shell into a ping tool", "T1059.004"),
    ("look_around", "Look around", "T1082, T1087.001, T1033, T1083"),
    ("writable_probe", "Find a writable folder, then check with a hex echo", "T1083"),
    ("busybox_tag", "Check for a real BusyBox", "T1497.001"),
    ("echo_marker", "Check that echo works", "T1497.001"),
    ("shell_test", "The SHELL_TEST check", "T1497.001"),
    ("download", "Download and run", "T1105, T1222.002, T1059.004"),
    ("remove_rivals", "Remove rival bots", "T1489"),
    ("clean_up", "Clean up", "T1070.004"),
]
usernames = safe_usernames(after)

rows = []
for key, step, attack in STEPS:
    if key == "login":
        lines, n_sessions, ips = len(sent_login(after)), len(sessions), sessions["ip_id"].nunique()
        example = "usernames such as " + ", ".join(usernames.index[:4])
    else:
        chosen = cmds[cmds[key]] if key in ("remove_rivals", "clean_up") else cmds[cmds["family"] == key]
        lines, n_sessions, ips = len(chosen), chosen["session"].nunique(), chosen["ip_id"].nunique()
        example = example_for(key, chosen["cmd"])
    rows.append({"key": key, "step": step, "ATT&CK": attack, "sessions": n_sessions,
                 "lines": lines, "IPs": ips, "example (defanged)": example})
playbook = pd.DataFrame(rows)

fig = go.Figure(go.Bar(
    x=playbook["sessions"], y=[f"{s}  ·  {a}" for s, a in zip(playbook["step"], playbook["ATT&CK"])],
    orientation="h", marker={"color": T["s1"]},
    text=playbook["sessions"], textposition="outside", textfont={"color": T["ink"]}, cliponaxis=False,
    customdata=playbook[["lines", "IPs"]],
    hovertemplate="%{y}<br>%{x} sessions, %{customdata[0]:,} lines, %{customdata[1]} IPs<extra></extra>",
))
style(fig, "What bots did after logging in, in attack order",
      "Sessions that took each step, Nov 3–6, with MITRE ATT&CK IDs. Most sessions took several steps.",
      height=600)
fig.update_layout(bargap=0.4)
fig.update_yaxes(autorange="reversed", showgrid=False, ticks="", showline=False, tickfont={"color": T["ink2"]})
fig.update_xaxes(title_text="Sessions", showline=True)
show(fig, "4-playbook")
other = cmds[cmds["family"] == "other"]
print(f"Not in a step: {len(other)} empty, non-text or garbled lines from {other['ip_id'].nunique()} IPs.")
display(playbook.drop(columns="key"))

## 5. Usernames tried

Ranked by how many IPs tried each name, so one bot repeating itself can't top the list.
Passwords are never shown.

In [ ]:
top_users = usernames.head(10)
fig = go.Figure(go.Bar(
    x=top_users["ips"], y=[html.escape(u) for u in top_users.index], orientation="h",
    marker={"color": T["s1"]}, text=top_users["ips"], textposition="outside",
    textfont={"color": T["ink"]}, cliponaxis=False, customdata=top_users[["attempts"]],
    hovertemplate="%{y}: tried by %{x} IPs, %{customdata[0]:,} attempts<extra></extra>",
))
style(fig, "The usernames bots tried most widely",
      "Nov 3–6, top 10 by number of IPs. Names tried by only one IP are left out, because a "
      "one-off could be a password typed into the wrong box.", height=460)
fig.update_layout(bargap=0.4)
fig.update_yaxes(autorange="reversed", showgrid=False, ticks="", showline=False, tickfont={"color": T["ink2"]})
fig.update_xaxes(title_text="IP addresses that tried it", showline=True)
show(fig, "5-usernames")
display(top_users)

## 6. How shell sessions ended

In [ ]:
ENDED = {"left": "Hung up", "timed_out": "Hit the 30-second limit", "exit": "Typed exit",
         "over_limit": "Went over a session limit"}
ended = sessions["ended"].value_counts().rename(index=ENDED)
total = int(ended.sum())

fig = go.Figure()
for (label, n), color in zip(ended.items(), [T["s1"], T["s2"], T["muted"], T["axis"]]):
    share = n / total
    fig.add_bar(
        y=["sessions"], x=[n], orientation="h", name=label, width=0.3,
        marker={"color": color, "line": {"color": T["surface"], "width": 2}},
        text=[f"  {n:,} ({share:.0%})"], textposition="inside", insidetextanchor="start",
        textfont={"color": "#ffffff"},
        hovertemplate=f"{label}: {n} sessions ({share:.0%})<extra></extra>",
    )
note = "None typed exit. " if "Typed exit" not in ended.index else ""
style(fig, f"How the {total} shell sessions ended",
      f"Nov 3–6. {note}Every visit has a 30-second limit, so a session that hit it held the bot for the full 30 seconds.",
      height=230)
fig.update_layout(barmode="stack", margin={"b": 16}, legend_traceorder="normal")
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
show(fig, "6-sessions-ended")
display(ended.rename("sessions").to_frame().assign(share=lambda d: (100 * d["sessions"] / total).round(1)))

## Check against README_REPORT.md

Every number the report takes from the logs, recomputed here. The few that come from elsewhere
(server health, data sent, test counts) aren't in the logs, so they aren't checked.

In [ ]:
creds = [line for line in (REPO / "deploy/host/credentials.txt").read_text().splitlines()
         if line.strip() and not line.lstrip().startswith("#")]
days = list(after.groupby(after["time"].dt.date))
day_silent = [pct(len(silent(g)), len(g), 1) for _, g in days]
downloads = cmds[cmds["family"] == "download"]["cmd"]
shell_test_only = sessions[sessions["commands"].map(lambda c: c == ["echo SHELL_TEST"])]
step = playbook.set_index("key")
checked = sessions[sessions["ran_check"]]
mirai_opening = ["enable", "system", "shell", "sh", "linuxshell"]

checks = [
    ("Hours live", 86, round(hours_live)),
    ("Connections", 2450, len(after)),
    ("IP addresses", 364, after["ip_id"].nunique()),
    ("Logins into the fake shell", 481, len(sessions)),
    ("IPs that got in", 152, sessions["ip_id"].nunique()),
    ("Commands", 1772, len(cmds)),
    ("Different commands", 74, cmds["cmd"].nunique()),
    ("Sessions that reached a download", 13, int(sessions["downloaded"].sum())),
    ("IPs that tried a download", 7, sessions[sessions["downloaded"]]["ip_id"].nunique()),
    ("Download addresses", 88, int(sessions["url_ids"].map(len).sum())),
    ("Unique download addresses", 7, len({u for us in sessions["url_ids"] for u in us})),
    ("Silent visits after", 284, len(silent(after))),
    ("Silent % after", 12, pct(len(silent(after)), len(after))),
    ("Factory-default logins accepted", 18, len(creds)),
    ("Sent a login", 2132, len(sent_login(after))),
    ("Sent a login %", 87, pct(len(sent_login(after)), len(after))),
    ("Sessions with a command", 414, int(with_command.sum())),
    ("Download share of sessions %", 2.7, pct(int(sessions["downloaded"].sum()), len(sessions), 1)),
    ("Sessions that hung up", 289, int((sessions["ended"] == "left").sum())),
    ("Sessions that hit the 30-second limit", 192, int((sessions["ended"] == "timed_out").sum())),
    ("Sessions that ran a real-shell check", 165, len(checked)),
    ("…and never downloaded", 161, int((~checked["downloaded"]).sum())),
    ("…and downloaded anyway", 4, int(checked["downloaded"].sum())),
    ("Sessions opening with enable, system, shell, sh, linuxshell", 54,
     int(sessions["commands"].map(lambda c: c[:5] == mirai_opening).sum())),
    ("Writable-folder probe variants", 25, cmds[cmds["family"] == "writable_probe"]["cmd"].nunique()),
    ("echo SHELL_TEST-only sessions", 47, len(shell_test_only)),
    ("…from IPs", 3, shell_test_only["ip_id"].nunique()),
    ("…on days", "Nov 3, Nov 4", ", ".join(sorted({f"{t:%b} {t.day}" for t in shell_test_only["time"]}))),
    ("Rival botnets named in the kill list", 10,
     len(RIVALS.search(cmds[cmds["remove_rivals"]]["cmd"].iloc[0])[1].split())),
    ("Droppers that passed telnet to their payload", 2,
     downloads[downloads.str.contains(r"\bsh (?:-s |\S+\.sh )telnet", regex=True)].nunique()),
    ("Every dropper has a fallback chain", True, bool(downloads.str.contains("||", regex=False).all())),
    ("Download ports", "80, 443, 8081, 9087",
     ", ".join(str(p) for p in sorted({p for ps in sessions["url_ports"] for p in ps if p}))),
    ("A dropper also used TFTP", True, bool(downloads.str.contains("tftp", regex=False).any())),
    ("root, admin, default and support are among the top usernames", True,
     {"root", "admin", "default", "support"} <= set(usernames.index[:10])),
    ("Before: visits", 520, len(before)),
    ("Before: silent", 425, len(silent(before))),
    ("Before: silent %", 82, pct(len(silent(before)), len(before))),
    ("Before: retry-pattern IPs", 11, len(retry_before)),
    ("Before without retry IPs: visits", 279, len(before_no_retry)),
    ("Before without retry IPs: silent", 184, len(silent(before_no_retry))),
    ("Before without retry IPs: silent %", 66, pct(len(silent(before_no_retry)), len(before_no_retry))),
    ("Before: sent a login", 81, len(sent_login(before))),
    ("Before: sent a login %", 16, pct(len(sent_login(before)), len(before))),
    ("Before: IP addresses", 117, before["ip_id"].nunique()),
    ("After: first 24 hours visits", 713, len(first_day)),
    ("After: first 24 hours silent", 99, len(silent(first_day))),
    ("After: first 24 hours silent %", 14, pct(len(silent(first_day)), len(first_day))),
    ("After: answered the telnet opening", 1994, int(after["telnet_answered"].sum())),
    ("After: answered the telnet opening %", 81, pct(int(after["telnet_answered"].sum()), len(after))),
    ("After: retry-pattern IPs on the busiest day", 0, max(len(retry_ips(g)) for _, g in days)),
    ("After: visits from the 11 retry IPs", 0, len(retry_after)),
    ("IPs seen before and after", 7, len(seen_both)),
    ("…their visits before", 109, len(b_both)),
    ("…silent before", 44, len(silent(b_both))),
    ("…silent % before", 40, pct(len(silent(b_both)), len(b_both))),
    ("…their visits after", 434, len(a_both)),
    ("…silent after", 35, len(silent(a_both))),
    ("…silent % after", 8, pct(len(silent(a_both)), len(a_both))),
    ("Lowest daily silent %", 8, round(min(day_silent))),
    ("Highest daily silent %", 13, round(max(day_silent))),
]
for key, lines, ips in [("menu_escape", 1082, 92), ("router_cli", 240, 25), ("ping_injection", 64, 48),
                        ("look_around", 177, 50), ("writable_probe", 80, 25), ("busybox_tag", 29, 11),
                        ("echo_marker", 13, 9), ("shell_test", 47, 3), ("download", 13, 7),
                        ("remove_rivals", 1, 1), ("clean_up", 1, 1)]:
    checks.append((f"Playbook: {step.loc[key, 'step']}: lines", lines, int(step.loc[key, "lines"])))
    checks.append((f"Playbook: {step.loc[key, 'step']}: IPs", ips, int(step.loc[key, "IPs"])))

report_check = pd.DataFrame(checks, columns=["number in README_REPORT.md", "report", "notebook"])
report_check["match"] = [r == n for r, n in zip(report_check["report"], report_check["notebook"])]
wrong = report_check[~report_check["match"]]
print(f"{report_check['match'].sum()} of {len(report_check)} numbers match the report."
      + ("" if wrong.empty else " These don't:"))
display(wrong if not wrong.empty else report_check)

## Save the charts as images

Each chart's camera button downloads a PNG. To write PNG and SVG files for all six at once, set
`EXPORT = True` and run everything. They go in `analysis/figures/`, which git ignores, named with
the theme, such as `1-funnel-light.png`. The website needs both themes, so run it once with
`THEME = "light"` and once with `THEME = "dark"`. Copy the files you want into the site or report
on purpose. The charts hold only counts and masked labels.

In [ ]:
EXPORT = False

if EXPORT:
    out = REPO / "analysis" / "figures"
    out.mkdir(exist_ok=True)
    for name, fig in FIGURES.items():
        # The theme is in the name, so a light run and a dark run can sit side by side.
        fig.write_image(out / f"{name}-{THEME}.png", width=1000, scale=2)
        fig.write_image(out / f"{name}-{THEME}.svg", width=1000)
    print(f"Wrote {len(FIGURES) * 2} {THEME} files to {out}")